# Data Generation & AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/05_data_generation_ai.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/05_data_generation_ai.ipynb)

Generate synthetic test data with referential integrity across tables, inject edge cases, validate quarantine coverage, and auto-infer contracts from CSVs.

In [ ]:
import os
import subprocess
import sys

# ── Install lakelogic ─────────────────────────────────────────────────
# Update the path below to match your local lakelogic checkout.
# On Colab (or if the path doesn't exist), falls back to PyPIuv
_LAKELOGIC_LOCAL = r"C:\_Personal\_SaaS\lakelogic"

if os.path.isdir(_LAKELOGIC_LOCAL):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", _LAKELOGIC_LOCAL, "-q"])
    print(f"\u2705 Installed lakelogic (editable) from {_LAKELOGIC_LOCAL}")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "lakelogic", "-q"])
    print("\u2705 Installed lakelogic from PyPI")

In [ ]:
import subprocess
import sys
import importlib
import urllib.request
import os

if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-q", "lakelogic[polars]"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

---
## 1. DataGenerator Basics — Synthetic Data From a Contract
**The Problem:** You need test data that matches your schema. Writing Faker scripts for every table is tedious and drifts out of sync with your contracts.

**The Solution:** `DataGenerator` reads your contract and generates realistic data — including controlled invalid rows for quarantine testing.

In [ ]:
contract = s.write_contract(
    """
version: 1.0.0
dataset: test_users
model:
  fields:
    - name: user_id
      type: integer
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: age
      type: integer
      min: 18
      max: 120
    - name: country
      type: string
      accepted_values: [US, GB, DE, FR, JP]
    - name: status
      type: string
      accepted_values: [active, inactive, suspended]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: valid_age
      sql: "age BETWEEN 18 AND 120"
    - name: valid_country
      sql: "country IN ('US','GB','DE','FR','JP')"
""",
    "05_data_generation_ai_demo/users.yaml",
)

gen = ll.DataGenerator(contract)
df = gen.generate(rows=1000, invalid_ratio=0.10)

In [ ]:
# The Proof
print(f"Generated {len(df)} rows with ~10% intentionally invalid")
display(df.head(10))

---
## 2. Custom AI-Steered Scenarios
**The Problem:** Heuristic or pure random data often fails to test very specific business logic states or regional subsets.

**The Solution:** Use `ai=True` alongside `ai_custom_scenario` to specifically guide the AI generator while keeping strict adherence to your schema boundaries.

In [ ]:
import os
import lakelogic as ll

# ── 1. Configure the AI Provider ──────────────────────────────────────
# LakeLogic supports: openai, anthropic, azure, Google Gemini, ollama, and anything
# LiteLLM supports.  Set your provider, model, and API key here.
#
# Uncomment the provider you want to use:

# -- OpenAI --
os.environ["LAKELOGIC_AI_PROVIDER"] = "openai"
os.environ["LAKELOGIC_AI_MODEL"]    = "gpt-4o-mini"
# os.environ["OPENAI_API_KEY"]      = "sk-..."

# -- Anthropic --
# os.environ["LAKELOGIC_AI_PROVIDER"] = "anthropic"
# os.environ["LAKELOGIC_AI_MODEL"]    = "claude-sonnet-4-20250514"
# os.environ["ANTHROPIC_API_KEY"]     = "sk-ant-..."

# -- Google Gemini --
# os.environ["LAKELOGIC_AI_PROVIDER"] = "google"
# os.environ["LAKELOGIC_AI_MODEL"]    = "gemini-2.5-flash"
# os.environ["GOOGLE_API_KEY"]         = "AIz....."

# -- Ollama (local, free) --
# os.environ["LAKELOGIC_AI_PROVIDER"] = "ollama"
# os.environ["LAKELOGIC_AI_MODEL"]    = "llama3"

AI_PROVIDER = os.getenv("LAKELOGIC_AI_PROVIDER", "google")
AI_MODEL    = os.getenv("LAKELOGIC_AI_MODEL", "gemini-2.5-flash")
AI_API_KEY  = os.getenv("GOOGLE_API_KEY", "")  # or ANTHROPIC_API_KEY etc.

# ── 2. Define the data contract ──────────────────────────────────────
scenario_contract = s.write_contract(
    """
version: 1.0.0
dataset: ecommerce_users
model:
  fields:
    - name: user_id
      type: integer
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: full_name
      type: string
      required: true
    - name: age
      type: integer
      min: 18
      max: 120
    - name: country
      type: string
      accepted_values: [US, GB, DE, FR, JP, BR, IN]
    - name: tier
      type: string
      accepted_values: [free, pro, enterprise]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: valid_age
      sql: "age BETWEEN 18 AND 120"
""",
    "05_data_generation_ai_demo/scenario_users.yaml",
)

# ── 3. Generate with a custom scenario ───────────────────────────────
# The scenario string is injected directly into the LLM prompt.
# The AI will generate realistic sample pools AND edge cases that
# respect your natural-language instructions.

gen = ll.DataGenerator(scenario_contract)

scenario_df = gen.generate(
    rows=20,
    invalid_ratio=0.20,
    ai=True,
    ai_provider=AI_PROVIDER,
    ai_model=AI_MODEL,
    ai_api_key=AI_API_KEY,
    ai_custom_scenario=(
        "Generate users who are French (FR) or Japanese (JP) only. "
        "All valid users should be enterprise-tier and over 60 years old. "
        "Use realistic French and Japanese names for the full_name field. "
        "For invalid edge cases, inject SQL injection strings into email "
        "and negative ages."
    ),
)

print(f"Generated {len(scenario_df)} rows with custom AI scenario")
display(scenario_df.head(10))

# ── 4. Validate through the pipeline ─────────────────────────────────
proc = ll.DataProcessor(scenario_contract, engine="polars")
good, bad = proc.run(scenario_df)

print(f"\nGood: {len(good)} | Quarantined: {len(bad)}")
s.assert_reconciliation(scenario_df, good, bad)
if len(bad) > 0:
    print("\nQuarantined rows (AI-generated edge cases):")
    display(bad.head(5))

---
## 3. Streaming Simulation -- Time-Windowed Batch Generation
**The Problem:** Your pipeline must handle incremental data arriving in time windows.
You need test data that simulates realistic ingestion patterns -- not just static dumps.

**The Solution:** `DataGenerator.generate_stream()` produces batches with monotonically
increasing timestamps, each confined to a configurable time window.


In [ ]:
# -- Streaming Simulation: time-windowed batch generation ----------
import lakelogic as ll
import polars as pl

stream_contract = s.write_contract(
    """
version: 1.0.0
dataset: streaming_events

model:
  fields:
    - name: event_id
      type: integer
    - name: event_ts
      type: timestamp
    - name: user_id
      type: integer
    - name: action
      type: string
      accepted_values: [page_view, click, purchase, signup]
""",
    "05_data_generation_ai_demo/streaming_events.yaml",
)

gen = ll.DataGenerator(stream_contract)

# Simulate 3 batches arriving every 15 minutes, 5 rows each
all_batches = []
for window_start, window_end, batch_df in gen.generate_stream(batches=3, interval_minutes=15, rows_per_batch=5):
    print(f"Window: {window_start} -> {window_end} | {len(batch_df)} rows")
    all_batches.append(batch_df)

df_stream = pl.concat(all_batches)
print(f"\nTotal: {len(df_stream)} events across 3 windows")
display(df_stream.select(["event_id", "event_ts", "action"]).head(10))
print("\n\u2705 Streaming simulation. Faker. $0 cost.")

---
## 4. Referential Integrity — FK/PK Consistency Across Tables
**The Problem:** You generate test customers and test orders separately. Half your order rows reference `customer_id` values that don't exist in the customers table. Your join tests fail for the wrong reasons.

**The Solution:** `DataGenerator.generate_related()` detects FK/PK relationships between contracts, generates parent tables first, then passes parent PKs into child tables so every foreign key is valid.

In [ ]:
# Define two related contracts: customers (parent) and orders (child)
customers_path = s.write_contract(
    """
version: 1.0.0
dataset: customers
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: tier
      type: string
      accepted_values: [free, pro, enterprise]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
  dataset_rules:
    - unique: customer_id
""",
    "05_data_generation_ai_demo/ri_customers.yaml",
)

orders_path = s.write_contract(
    """
version: 1.0.0
dataset: orders
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: customer_id
      type: integer
      required: true
      foreign_key:
        contract: customers
        column: customer_id
    - name: amount
      type: float
      required: true
    - name: status
      type: string
      accepted_values: [pending, shipped, delivered]
quality:
  row_rules:
    - name: positive_amount
      sql: "amount > 0"
    - referential_integrity:
        field: customer_id
        contract: customers
        column: customer_id
        severity: critical
  dataset_rules:
    - unique: order_id
""",
    "05_data_generation_ai_demo/ri_orders.yaml",
)

# Generate both tables with referential integrity
related = ll.DataGenerator.generate_related(
    contracts={
        "customers": customers_path,
        "orders": orders_path,
    },
    rows={"customers": 50, "orders": 200},
    invalid_ratio=0.05,
)

customers_df = related["customers"]
orders_df = related["orders"]

In [ ]:
# The Proof — every order.customer_id exists in customers.customer_id
import polars as pl

parent_ids = set(customers_df["customer_id"].to_list())
child_ids = set(orders_df["customer_id"].to_list())
orphans = child_ids - parent_ids

print(f"Customers: {len(customers_df)} rows, {len(parent_ids)} unique IDs")
print(f"Orders:    {len(orders_df)} rows")
print(f"Orphan FKs: {len(orphans)}")
print()

# Show the FK distribution
fk_counts = orders_df.group_by("customer_id").len().sort("len", descending=True)
print("Orders per customer (top 5):")
display(fk_counts.head(5))
print("\n50 customers, 200 orders — every FK references a real parent row.")

In [ ]:
# Validate both tables through their contracts
proc_c = ll.DataProcessor(customers_path, engine="polars")
good_c, bad_c = proc_c.run(customers_df)

proc_o = ll.DataProcessor(orders_path, engine="polars")
good_o, bad_o = proc_o.run(orders_df)

print("Customers:")
s.assert_reconciliation(customers_df, good_c, bad_c)
print("\nOrders:")
s.assert_reconciliation(orders_df, good_o, bad_o)
print("\nBoth tables validated. Referential integrity preserved end-to-end.")

---
## 5. Edge Case Injection — SQL Injection, Boundary Values
**The Problem:** Your tests use happy-path data. SQL injection strings, unicode edge cases, and boundary values never get tested — until production.

**The Solution:** `DataGenerator` with `invalid_ratio` injects realistic attack vectors and boundary violations.

In [ ]:
# Generate with higher invalid ratio to see more edge cases
gen = ll.DataGenerator(contract)
edge_df = gen.generate(rows=500, invalid_ratio=0.20)

# Run through the pipeline
proc = ll.DataProcessor(contract, engine="polars")
good, bad = proc.run(edge_df)

In [ ]:
# The Proof — quarantine caught edge cases
print(f"Source: {len(edge_df)} | Good: {len(good)} | Quarantined: {len(bad)}")
s.assert_reconciliation(edge_df, good, bad)
print()
print("Quarantined rows (sample of edge cases caught):")
display(bad.head(10))
print("\nSQL injection, boundary values, type violations — all caught by the contract.")

---
## 6. Full Test Coverage — Generate, Run, Prove
**The Problem:** How do you prove your quality rules actually work? Manual test data is incomplete and goes stale.

**The Solution:** Generate targeted invalid data, run it through the pipeline, assert every bad row was quarantined.

In [ ]:
# Generate 100% invalid data — every row should be quarantined
data_test_cov = gen.generate(rows=200, invalid_ratio=1.0)

proc = ll.DataProcessor(contract, engine="polars")
good, bad = proc.run(data_test_cov)

In [ ]:
# The Proof
print(f"Source: {len(data_test_cov)} (data_test_cov)")
print(f"Good:   {len(good)}")
print(f"Bad:    {len(bad)}")
print()
quarantine_rate = len(bad) / len(data_test_cov) * 100 if len(data_test_cov) > 0 else 0
print(f"Quarantine rate: {quarantine_rate:.1f}%")
print(f"\nYour quality rules caught {'all' if len(good) == 0 else 'most'} invalid rows.")

---
## 7. `infer_contract` — Contract From a CSV in 30 Seconds
**The Problem:** You have 50 CSVs and no contracts. Writing YAML by hand for each one takes days.

**The Solution:** Point `infer_contract` at a file. It detects types, PII fields, and suggests quality rules.

In [ ]:
from lakelogic.core.bootstrap import infer_contract

# Create a sample CSV
sample = pl.DataFrame(
    {
        "order_id": list(range(1, 101)),
        "customer_email": [f"user{i}@example.com" for i in range(1, 101)],
        "amount": [round(i * 9.99, 2) for i in range(1, 101)],
        "country": ["US", "GB", "DE", "FR", "JP"] * 20,
        "created_at": ["2026-01-15"] * 100,
    }
)
sample.write_csv("sample_orders.csv")

# Infer a contract from the CSV
draft = infer_contract("sample_orders.csv", title="Inferred Orders")

In [ ]:
# The Proof
draft.show()
print("\nContract inferred in seconds. PII detected. Types resolved. Ready to customise.")

---
## 8. Unstructured Processing — Contract-Driven Extraction
**The Problem:** You have PDFs, scanned images, or free-text. Regex breaks on format changes. Custom parsers drift from your schema.

**The Solution:** Declare *what* to extract in `model.fields` and *how* in `extraction:`. LakeLogic picks the right library, extracts, validates, and materialises — one call.

| Provider | Extra | Input | Use Case |
|----------|-------|-------|----------|
| `local` (pdfplumber) | `lakelogic[extraction-ocr]` | PDF | Table + text, $0, no API key |
| `spacy` | `lakelogic[nlp]` | Free text | NER + classification, $0, local |
| `rapidocr` | `lakelogic[extraction-ocr]` | Scanned image | ONNX OCR, pure Python, no torch |
| `openai` / `anthropic` | `lakelogic[ai]` | Any | LLM prompting with structured output |

In [ ]:
# Generate demo assets — invoice PDF and support tickets
import os
import shutil
import tempfile
import subprocess
import sys
import polars as pl

DEMO_DIR = os.path.join(tempfile.gettempdir(), "lakelogic_extraction_demo")
shutil.rmtree(DEMO_DIR, ignore_errors=True)
os.makedirs(DEMO_DIR, exist_ok=True)

try:
    from fpdf import FPDF
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fpdf2"])
    from fpdf import FPDF

# Build a realistic invoice PDF with a table of line items
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", "B", 20)
pdf.cell(0, 15, "INVOICE", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.set_font("Helvetica", "", 11)
pdf.cell(0, 8, "Acme Corp | 500 Market St, San Francisco, CA 94105", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.ln(4)
pdf.set_font("Helvetica", "B", 11)
pdf.cell(95, 8, "Invoice #: INV-2026-0042")
pdf.cell(95, 8, "Date: April 15, 2026", new_x="LMARGIN", new_y="NEXT", align="R")
pdf.cell(95, 8, "Bill To: Globex Corporation")
pdf.cell(95, 8, "Due: May 15, 2026", new_x="LMARGIN", new_y="NEXT", align="R")
pdf.ln(4)
pdf.set_fill_color(240, 240, 240)
pdf.set_font("Helvetica", "B", 10)
for col, w in [("Description", 90), ("Hours", 30), ("Rate", 35), ("Amount", 35)]:
    is_last = col == "Amount"
    pdf.cell(w, 8, col, border=1, fill=True, align="C", **(dict(new_x="LMARGIN", new_y="NEXT") if is_last else {}))
pdf.set_font("Helvetica", "", 10)
LINE_ITEMS = [
    ("Data Platform Architecture", 40, 200, 8000),
    ("Pipeline Development", 60, 175, 10500),
    ("Quality Assurance & Testing", 20, 150, 3000),
]
for desc, hrs, rate, amt in LINE_ITEMS:
    pdf.cell(90, 7, desc, border=1)
    pdf.cell(30, 7, str(hrs), border=1, align="C")
    pdf.cell(35, 7, f"${rate:.2f}", border=1, align="C")
    pdf.cell(35, 7, f"${amt:,.2f}", border=1, align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "B", 12)
pdf.cell(155, 10, "Total:", align="R")
pdf.cell(35, 10, "$21,500.00", align="C", new_x="LMARGIN", new_y="NEXT")

PDF_PATH = os.path.join(DEMO_DIR, "demo_invoice.pdf")
pdf.output(PDF_PATH)

# Five support tickets for classification
tickets = pl.DataFrame(
    {
        "ticket_id": [1001, 1002, 1003, 1004, 1005],
        "ticket_body": [
            "I was charged $2,500 for a subscription I cancelled. Billing error ongoing since March.",
            "Order #4521 shipped to London but I live in Manchester. Please redirect via FedEx.",
            "New MacBook Pro has a cracked screen. Returns team arranged replacement immediately.",
            "Enterprise license for 500 seats expires next week. Discuss renewal and 200 more seats.",
            "API returning 500 errors since 2pm. Blocking our entire production pipeline. Fix now.",
        ],
    }
)


print(f"PDF invoice : {PDF_PATH}")

In [ ]:
# -- Flavour 1: PDF Invoice → pdfplumber --------------------------
# The contract defines model fields, extraction provider, and quality rules.
# extract_file() handles all parsing — no manual logic in the notebook.

from lakelogic.engines.llm import extract_file
from lakelogic.core.models import ExtractionConfig

pdf_contract = s.write_contract(
    """
version: 1.0.0
dataset: invoice_line_items

model:
  fields:
    # -- Metadata (extracted from page text via regex) ----------
    - name: invoice_number
      type: string
      required: true
      extraction_task: metadata
      extraction_examples: ['Invoice #:\\s*(\\S+)']
    - name: vendor
      type: string
      extraction_task: metadata
      extraction_examples: ['INVOICE\\n(.+?)\\n']
    - name: date
      type: string
      extraction_task: metadata
      extraction_examples: ['Date:\\s*(.+?)\\n']
    - name: bill_to
      type: string
      extraction_task: metadata
      extraction_examples: ['Bill To:\\s*(.+?)\\s+Due:']
    - name: due_date
      type: string
      extraction_task: metadata
      extraction_examples: ['Due:\\s*(.+?)(?:\\n|$)']

    # -- Table rows (matched by column header name) ------------
    - name: description
      type: string
      required: true
    - name: hours
      type: integer
    - name: rate
      type: string
    - name: amount
      type: string

extraction:
  provider: pdfplumber

quality:
  row_rules:
    - name: has_description
      sql: "description IS NOT NULL AND description != ''"
    - name: has_invoice_number
      sql: "invoice_number IS NOT NULL"
""",
    "05_data_generation_ai_demo/invoice_line_items.yaml",
)

# -- Extract -------------------------------------------------------
import yaml

contract_dict = yaml.safe_load(open(pdf_contract))
ext_config = ExtractionConfig(
    provider=contract_dict["extraction"]["provider"],
    output_schema=contract_dict["model"]["fields"],
)
rows = extract_file(PDF_PATH, ext_config)

# -- Materialize --------------------------------------------------
import polars as pl

df_invoice = pl.DataFrame(rows)


display(df_invoice)
print(f"\n\u2705 PDF \u2192 {len(rows)} line items + metadata. pdfplumber. $0 cost.")

In [ ]:
# ── Side-by-side: Raw PDF text vs Extracted Table ─────────────────
import pdfplumber

with pdfplumber.open(PDF_PATH) as doc:
    raw_text = doc.pages[0].extract_text()

print("RAW PDF TEXT".center(60, "\u2500"))
print(raw_text)
print()
print("EXTRACTED TABLE".center(60, "\u2500"))
display(df_invoice.drop([c for c in df_invoice.columns if c.startswith("_")]))
print(f"\n\u2500 Source: {PDF_PATH}")

In [ ]:
# -- Flavour 2: Support Tickets -> spaCy NER + Classification ----
# spaCy extracts entities + classifies text locally at production speed.

from lakelogic.engines.llm import extract_row

nlp_contract = s.write_contract(
    """
version: 1.0.0
dataset: enriched_tickets

model:
  fields:
    - name: persons
      type: string
      extraction_task: ner
    - name: organizations
      type: string
      extraction_task: ner
      extraction_examples: [ORG]
    - name: category
      type: string
      extraction_task: classification
      accepted_values: [billing, shipping, product, enterprise, outage]
    - name: sentiment
      type: string
      extraction_task: sentiment

extraction:
  provider: spacy
  model: en_core_web_md       # medium model -- better NER than sm
  text_column: ticket_body

quality:
  row_rules:
    - name: has_sentiment
      sql: "sentiment IS NOT NULL"
""",
    "05_data_generation_ai_demo/enriched_tickets.yaml",
)

# -- Extract each ticket -----------------------------------------------
contract_dict = yaml.safe_load(open(nlp_contract))
ext_config = ExtractionConfig(
    provider=contract_dict["extraction"]["provider"],
    text_column=contract_dict["extraction"].get("text_column", "text"),
    output_schema=contract_dict["model"]["fields"],
)

enriched = [extract_row(row, ext_config) for row in tickets.to_dicts()]

# -- Materialize -------------------------------------------------------
df_tickets = pl.DataFrame(enriched)

display_cols = ["ticket_id", "persons", "organizations", "category", "sentiment"]

print("RAW ticket TEXT".center(60, "\u2500"))
print(tickets)
print()
print("EXTRACTED TABLE".center(60, "\u2500"))

display(df_tickets.select([c for c in display_cols if c in df_tickets.columns]))
print(f"\n\u2705 {len(enriched)} tickets enriched. spaCy. $0 cost.")

---
## 9. Automated Run Logs — Structured Pipeline Observability
**The Problem:** Pipelines fail silently. Row counts drift. Quarantine tables fill up. But you only find out when a dashboard is empty.

**The Solution:** Every pipeline run automatically emits a structured, comprehensive run log. These logs can be written out to a Delta table, making your entire data operations history immediately queryable.

In [ ]:
import lakelogic as ll
from lakelogic.core.run_log import write_run_log
import polars as pl
import duckdb
import os
import tempfile

# ── 1. Configure the pipeline to write actual run logs to DuckDB ──────
LOG_DIR = os.path.join(tempfile.gettempdir(), "lakelogic_logs")
os.makedirs(LOG_DIR, exist_ok=True)
DB_PATH = os.path.join(LOG_DIR, "run_logs.duckdb").replace("\\", "/")

# Remove stale DB so we start fresh each demo run
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

log_contract_path = s.write_contract(
    f"""
version: 1.0.0
dataset: automated_log_demo
metadata:
  run_log_table: pipeline_run_logs
  run_log_backend: duckdb
  run_log_database: "{DB_PATH}"
model:
  fields:
    - name: user_id
      type: integer
    - name: age
      type: integer
quality:
  row_rules:
    - name: valid_age
      sql: "age >= 18"
""",
    "05_data_generation_ai_demo/automated_log_demo.yaml",
)

# Load as a proper DataContract object (write_run_log needs .metadata)
log_contract = ll.DataContract.from_yaml(log_contract_path)

# ── 2. Run the pipeline a few times with varying data quality ─────────
proc = ll.DataProcessor(log_contract)


def simulate_run(data):
    proc.run(pl.DataFrame(data))
    # Status is normally set by the pipeline runner after materialize;
    # in standalone mode we stamp it manually.
    report = proc.last_report
    quarantined = (report.get("counts") or {}).get("quarantined", 0)
    report["status"] = "warning" if quarantined else "success"
    write_run_log(report, log_contract)


# Run 1: Perfect data
simulate_run({"user_id": [1, 2], "age": [25, 30]})

# Run 2: One invalid row (age < 18)
simulate_run({"user_id": [3, 4], "age": [15, 40]})

# Run 3: All invalid rows
simulate_run({"user_id": [5, 6], "age": [10, 12]})

# ── 3. Query the actual Run Logs telemetry table ──────────────────────
con = duckdb.connect(DB_PATH, read_only=True)
query = """
  SELECT 
      run_id,
      contract,
      dataset, 
      counts_source, 
      counts_good, 
      counts_quarantined, 
      quarantine_ratio,
      status,
      start_time, 
      end_time,
      run_duration_seconds
  FROM pipeline_run_logs
"""
logs_df = con.execute(query).pl()
con.close()

print("AUTOMATED RUN LOGS (Queried from actual DuckDB backend):")
display(logs_df)

print(f"\n\u2705 {len(logs_df)} runs captured. BI tools can connect directly to: {DB_PATH}")

## What You Just Saw

| # | Feature | How |
|---|---------|-----|
| 1 | **Synthetic Data** | `DataGenerator` generates row-data from pure YAML contracts |
| 2 | **AI-Steered Scenarios** | Steer LLM generators natively using `ai_custom_scenario` prompts |
| 3 | **Streaming Simulation** | `generate_stream()` produces time-windowed batches for incremental testing |
| 4 | **Referential Integrity** | `generate_related()` natively matches Primary & Foreign Keys |
| 5 | **Edge Case Injection** | `invalid_ratio` simulates anomalies to verify your quarantine rules |
| 6 | **Test Coverage** | Full pipeline runs mechanically prove your rules filter bad rows |
| 7 | **AI Contract Onboarding** | `infer_contract` turns any CSV into a typed YAML contract instantly |
| 8 | **Unstructured Extraction** | `pdfplumber` (PDF) and `spaCy` (NER) extract structured data locally -- $0 cost |
| 9 | **Automated Run Logs** | Every run is stored as queryable structured telemetry |



---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.